# Entrenar YOLO propio — genérico (cualquier partido)

Entrena un YOLO11s con **nuestros** fotogramas (268 imágenes, 214 train / 54 val, clases: `0=football`, `1=player`).

**El modelo es genérico:** solo detecta balón y jugador, sin equipos ni colores. Los equipos se distinguen después por color de camiseta en `video_track.py`, así que vale para cualquier partido, estadio o luz.

**Requisitos:** GPU en Colab (Entorno > Cambiar tipo de entorno > GPU T4). El dataset ya está etiquetado:
- Etiquetas + `data.yaml` en el repo GitHub (`dataset/`).
- Imágenes en Drive, carpeta **Git Hub** (`dataset_yolo_propio_images.zip`).

**Composición (268 imágenes):** 210 del partido Cirbonero B vs Milagres (1 cada 30 s, todo el partido) + 58 de la 2ª mitad del mismo partido (luz oscura, atardecer/noche, 1 cada 45 s). El split train/val (80/20) está estratificado por fuente, así que val tiene de las dos luces.

**Para hacerlo aún más genérico (opcional):** la celda 4 descarga el dataset público GemsLabs-football-26 de Roboflow (276 imágenes de varios partidos, CC BY 4.0) y lo fusiona. Necesitas una cuenta gratuita en Roboflow y tu API key. Si la dejas vacía, el entrenamiento usa solo las 268 propias y funciona igual.

Al final el modelo queda en Drive (`Mi unidad/Git Hub/modelo_propio/best.pt`) para usarlo en `match.py` / `video_track.py`.

In [ ]:
# 1. Instalar ultralytics (y roboflow, solo para el paso opcional de la celda 4)
!pip install -q ultralytics roboflow

In [ ]:
# 2. Montar Drive y preparar el dataset
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, shutil

BASE = '/content/match_view'
os.makedirs(BASE, exist_ok=True)

# Imágenes: zip en Drive (carpeta "Git Hub")
ZIP = '/content/drive/MyDrive/Git Hub/dataset_yolo_propio_images.zip'
with zipfile.ZipFile(ZIP) as z:
    z.extractall(f'{BASE}/dataset')
print('imágenes:', len(os.listdir(f'{BASE}/dataset/images/train')) + len(os.listdir(f'{BASE}/dataset/images/val')))

In [ ]:
# 3. Bajar etiquetas + data.yaml del repo
!git clone -q https://github.com/Josen-IG/Match_view.git /content/repo_tmp

import shutil, os
# etiquetas (labels) del repo -> dataset/
for split in ('train', 'val'):
    src = f'/content/repo_tmp/dataset/labels/{split}'
    dst = f'{BASE}/dataset/labels/{split}'
    os.makedirs(dst, exist_ok=True)
    for f in os.listdir(src):
        shutil.copy(os.path.join(src, f), os.path.join(dst, f))
# data.yaml del repo (apunta a images/train y images/val)
shutil.copy('/content/repo_tmp/dataset/data.yaml', f'{BASE}/dataset/data.yaml')
print(open(f'{BASE}/dataset/data.yaml').read())
print('labels train:', len(os.listdir(f'{BASE}/dataset/labels/train')), '| val:', len(os.listdir(f'{BASE}/dataset/labels/val')))

In [ ]:
# 4. OPCIONAL: añadir el dataset público GemsLabs-football-26 (Roboflow, CC BY 4.0)
# 276 imágenes de VARIOS partidos (distintos estadios, luces y cámaras): es lo que más
# generaliza el modelo. Necesitas una cuenta gratuita en Roboflow y tu API key:
# https://app.roboflow.com/settings/api  (la key solo se usa aquí, no se guarda).
# Si la dejas vacía, se salta este paso y se entrena solo con las 268 propias.
ROBOFLOW_API_KEY = ''  # <-- pega aquí tu API key de Roboflow, o déjalo vacío

if ROBOFLOW_API_KEY:
    from roboflow import Roboflow
    import yaml
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    ds = rf.workspace('ssss-1eqnb').project('gemslabs-football-26').version(1).download('yolov11')
    rf_dir = ds.location
    # data.yaml de Roboflow: mapear sus clases a las nuestras por NOMBRE (0=football, 1=player)
    with open(os.path.join(rf_dir, 'data.yaml')) as f:
        rf_yaml = yaml.safe_load(f)
    rf_names = rf_yaml['names']  # p. ej. {0: 'football', 1: 'player'}
    ours = {'football': 0, 'player': 1}
    remap = {}
    for idx, name in rf_names.items():
        if str(name).lower() in ours:
            remap[int(idx)] = ours[str(name).lower()]
    print('clases Roboflow:', rf_names, '-> mapeo:', remap)
    # Fusionar: train de Roboflow -> nuestro train, valid -> nuestro val (prefijo rf_ para no chocar nombres)
    for rf_split, our_split in (('train', 'train'), ('valid', 'val')):
        img_src = os.path.join(rf_dir, rf_split, 'images')
        lbl_src = os.path.join(rf_dir, rf_split, 'labels')
        n = 0
        for img in os.listdir(img_src):
            stem = os.path.splitext(img)[0]
            lbl = os.path.join(lbl_src, stem + '.txt')
            if not os.path.exists(lbl):
                continue
            shutil.copy(os.path.join(img_src, img), f'{BASE}/dataset/images/{our_split}/rf_{img}')
            # reescribir etiquetas con las clases remapeadas
            out_lines = []
            with open(lbl) as f:
                for line in f:
                    parts = line.strip().split()
                    if not parts:
                        continue
                    c = int(float(parts[0]))
                    if c not in remap:
                        continue  # clase que no usamos (p. ej. referee si la hubiera)
                    out_lines.append(' '.join([str(remap[c])] + parts[1:]))
            with open(f'{BASE}/dataset/labels/{our_split}/rf_{stem}.txt', 'w') as f:
                f.write('\n'.join(out_lines) + ('\n' if out_lines else ''))
            n += 1
        print(f'Roboflow {rf_split} -> {our_split}: {n} imágenes añadidas')
    print('TOTAL train:', len(os.listdir(f'{BASE}/dataset/images/train')), '| val:', len(os.listdir(f'{BASE}/dataset/images/val')))
else:
    print('Sin API key: se entrena solo con las 268 imágenes propias.')

In [ ]:
# 5. Entrenar
# yolo11s.pt = base (misma arquitectura YOLO, pesos propios tras entrenar).
# imgsz 960 = el mismo que usa el análisis; batch 16 en T4 (si da OOM, baja a 8).
# epochs 50 con paciencia 10: el dataset es pequeño (214 train), no conviene mucho más.
# Augmentación fuerte para generalizar a cualquier partido/estadio/luz: variación de
# color y brillo (HSV), rotación/escala/traslación, volteo horizontal y mosaic/mixup.
from ultralytics import YOLO

model = YOLO('yolo11s.pt')
model.train(
    data=f'{BASE}/dataset/data.yaml',
    epochs=50,
    imgsz=960,
    batch=16,
    patience=10,
    project=f'{BASE}/runs',
    name='yolo_propio',
    device=0,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=5, translate=0.1, scale=0.5, shear=2.0,
    perspective=0.0, flipud=0.0, fliplr=0.5,
    mosaic=1.0, mixup=0.1, copy_paste=0.0,
)

In [ ]:
# 6. Validar y ver el resultado
from ultralytics import YOLO
best = YOLO(f'{BASE}/runs/yolo_propio/weights/best.pt')
metrics = best.val(data=f'{BASE}/dataset/data.yaml', imgsz=960)
print('mAP50:', round(metrics.box.map50, 3))
print('mAP50-95:', round(metrics.box.map, 3))
print('mAP50 por clase:', {k: round(v, 3) for k, v in metrics.box.maps.items()} if hasattr(metrics.box, 'maps') else 'ver results.csv')

In [ ]:
# 7. Guardar el modelo en Drive (para usarlo en match.py / video_track.py)
import shutil, os
OUT = '/content/drive/MyDrive/Git Hub/modelo_propio'
os.makedirs(OUT, exist_ok=True)
shutil.copy(f'{BASE}/runs/yolo_propio/weights/best.pt', f'{OUT}/best.pt')
shutil.copy(f'{BASE}/runs/yolo_propio/weights/last.pt', f'{OUT}/last.pt')
print('guardado en:', OUT)
print('tamaño best.pt (MB):', round(os.path.getsize(f'{OUT}/best.pt') / 1e6, 1))

## Usarlo en el análisis

En `match.py` / `video_track.py`, cambia el modelo a la ruta de `best.pt` (descargado de Drive) y vuelve a analizar. Si el balón mejora mucho, se puede afinar con más épocas o más fotogramas etiquetados.

**Nota:** el modelo entrena solo `football` y `player`. El árbitro (clase 2) está pendiente de etiquetar a mano; hasta entonces el árbitro se clasifica por color, como ahora.

**Para hacerlo aún más genérico en el futuro:** añade fotogramas de otros partidos (extrae 1 cada 30-45 s, auto-etiquétalos con el modelo GemsLabs `custom_model.pt` + YOLO11s como se hizo con estos, y vuelve a entrenar). Cuantos más estadios/luces, mejor generaliza.